# TARGE — Prismatic VLM (Colab)

End-to-end pipeline: setup → install → login → paths → data → overview → train → eval.


## 1. Setup: GPU check + mount Drive


In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')


## 2. Install dependencies


In [ ]:
%cd /content/drive/MyDrive/targe-prismatic-vlms
!pip install -e . --quiet
# !pip install flash-attn --no-build-isolation --quiet  # optional speedup on A100/L4


## 3. Logins (Hugging Face, W&B)


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

# HF
hf_token = userdata.get('hf_token')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

# W&B (optional — only needed if --trackers includes wandb)
try:
    wandb_token = userdata.get('wandb_token')
    os.environ["WANDB_API_KEY"] = wandb_token
except Exception:
    print("No wandb_token secret found — skipping W&B login.")


## 4. Configure filepaths

All paths live here. Drive = persistent (slow); local = fast scratch on the Colab VM.


In [ ]:
import os
from pathlib import Path

# --- Drive (persistent) ---
DRIVE_ROOT     = Path("/content/drive/MyDrive/targe-prismatic-vlms")
DRIVE_DATA     = DRIVE_ROOT / "data"                 # source-of-truth dataset
DRIVE_RUNS     = DRIVE_ROOT / "runs"                 # checkpoints + logs (persisted)
DRIVE_HF_CACHE = DRIVE_ROOT / "hf_cache"             # HF model cache (persisted)

# --- Local (fast scratch on Colab VM) ---
LOCAL_ROOT = Path("/content/targe")
LOCAL_DATA = LOCAL_ROOT / "data"                     # data copied here for fast IO

# Dataset subpath (relative to DRIVE_DATA / LOCAL_DATA)
DATASET_SUBDIR = Path("download/llava-laion-cc-sbu-558k")
CHAT_JSON_NAME = "chat.pruned.json"                  # pruned to images that exist on disk

for p in (DRIVE_DATA, DRIVE_RUNS, DRIVE_HF_CACHE, LOCAL_ROOT, LOCAL_DATA):
    p.mkdir(parents=True, exist_ok=True)

# HF cache on Drive (model weights survive Colab session resets)
os.environ["HF_HOME"] = str(DRIVE_HF_CACHE)

# Repo lives on Drive, so runs/ already persists — no symlink needed.
REPO_ROOT = DRIVE_ROOT

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"DRIVE_DATA : {DRIVE_DATA}")
print(f"LOCAL_DATA : {LOCAL_DATA}")
print(f"DRIVE_RUNS : {DRIVE_RUNS}")
print(f"HF_HOME    : {os.environ['HF_HOME']}")


## 5. Move data from Drive to Colab local disk

Training off Drive is ~10–100x slower than local SSD. Copy once per session.


In [ ]:
import shutil, time

SRC = DRIVE_DATA / DATASET_SUBDIR
DST = LOCAL_DATA / DATASET_SUBDIR
DST.parent.mkdir(parents=True, exist_ok=True)

if DST.exists() and any(DST.iterdir()):
    print(f"[skip] {DST} already populated.")
else:
    print(f"[copy] {SRC} -> {DST}")
    t0 = time.time()
    # rsync is the most robust way to move many small files off Drive
    !rsync -ah --info=progress2 "{SRC}/" "{DST}/"
    print(f"[done] took {time.time() - t0:.1f}s")

# Sanity: link the script's expected ./data dir to LOCAL_DATA
%cd /content/drive/MyDrive/targe-prismatic-vlms
!rm -rf data && ln -s {LOCAL_DATA} data
!ls -la data/download/


## 6. Dataset overview


In [ ]:
import json
from collections import Counter

dataset_dir = LOCAL_DATA / DATASET_SUBDIR
chat_path   = dataset_dir / CHAT_JSON_NAME

# File counts
n_files = sum(1 for _ in dataset_dir.rglob("*") if _.is_file())
n_imgs  = sum(1 for p in dataset_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})

# Chat-json stats
with open(chat_path) as f:
    chat = json.load(f)

n_entries     = len(chat)
n_with_image  = sum(1 for e in chat if e.get("image"))
n_text_only   = n_entries - n_with_image
turns_per_ex  = Counter(len(e.get("conversations", [])) for e in chat)

print(f"Dataset dir       : {dataset_dir}")
print(f"Total files       : {n_files:,}")
print(f"  image files     : {n_imgs:,}")
print(f"Chat JSON         : {chat_path.name}")
print(f"  total entries   : {n_entries:,}")
print(f"  with image      : {n_with_image:,}")
print(f"  text-only       : {n_text_only:,}")
print(f"  turns/example   : {dict(sorted(turns_per_ex.items()))}")

# This dataset has no formal train/val split — alignment uses all of it as train.
print("\nSplit             : train-only (no held-out val in llava-laion-cc-sbu-558k)")


## 7. Train (align stage)


In [ ]:
%cd /content/drive/MyDrive/targe-prismatic-vlms
!torchrun --standalone --nnodes 1 --nproc-per-node 1 scripts/pretrain.py \
  --model.type "targe-smollm2-360m-clipb-224px" \
  --stage align \
  --model.align_per_device_batch_size 14 \
  --model.align_global_batch_size 14 \
  --model.align_learning_rate 1e-4 \
  --model.align_max_steps 500 \
  --dataset.type "llava-v15" \
  --run_id "targe-smollm2-next" \
  --trackers '[jsonl,wandb]' \
  --wandb_entity nbusich-northeastern-university \
  --wandb_project targe


## 8. Eval (interactive generation against a checkpoint)


In [ ]:
RUN_ID = "targe-smollm2-next"
CKPT_DIR = DRIVE_RUNS / RUN_ID

# Most recent checkpoint
ckpts = sorted((CKPT_DIR / "checkpoints").glob("*.pt"))
assert ckpts, f"No checkpoints found in {CKPT_DIR / 'checkpoints'}"
LATEST = ckpts[-1]
print(f"Evaluating: {LATEST}")


In [ ]:
%cd /content/drive/MyDrive/targe-prismatic-vlms
!python scripts/generate.py --model_path "{CKPT_DIR}"
